# Suite DEF — Definitions

What a program declares, organized by meaning (`mbse.Programs.Framework.Definitions`): entities in scopes, with
lookup through transparent scopes, using-directives and bases. `Ccpp.Definitions.define` builds them from a C or C++
translation unit. Each case checks the entities, their declarations and definitions, and what names resolve to.

In [ ]:
from mbse.Programs.Ccpp import Ccpp20, Definitions as D, Syntax as S
from mbse.Programs.Ccpp._Standard import CcppStandard
from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Framework.Definitions import Entity, Program, Scope

CPP26 = CcppStandard(2026)


def define(text, standard=CPP26):
    unit = standard.parse(text)
    return unit, D.define(unit)


def listing(program):
    return [(e.kind, e.qualified_name(), e.signature) for e in program.entities()]


def names(unit, spelling):
    return [n for n in F.walk(unit) if isinstance(n, S.Identifier) and n.spelling == spelling]

## DEF-01 · Entities, scopes and lookup, on scopes built by hand

In [ ]:
root = Scope("namespace", None, None)
outer = Entity("namespace", "outer", root)
outer.scope = Scope("namespace", outer, root)
root.declare(outer)
inner = Entity("class", "Inner", outer.scope)
inner.scope = Scope("class", inner, outer.scope)
outer.scope.declare(inner)
field = Entity("field", "x", inner.scope)
inner.scope.declare(field)
assert field.qualified_name() == "outer::Inner::x" and repr(field) == "<field outer::Inner::x>"
assert repr(inner.scope) == "<class scope outer::Inner>" and repr(root) == "<namespace scope>"
unnamed = Entity("namespace", None, root)
assert repr(unnamed) == "<namespace (unnamed)>" and unnamed.qualified_name() == ""
dotted = Scope("module", None, None, separator=".")
klass = Entity("class", "A", dotted)
klass.scope = Scope("class", klass, dotted)
dotted.declare(klass)
method = klass.scope.declare(Entity("function", "b", klass.scope))
assert klass.scope.separator == "." and method.qualified_name() == "A.b" and Program(dotted).lookup("A.b") == [method]
assert Entity("variable", "free", None).qualified_name() == "free" and root.separator == "::"
alias = Entity("namespace alias", "o", root)
alias.target = outer
root.declare(alias)
again = Entity("namespace alias", "p", root)
again.target = alias
root.declare(again)
assert again.resolved() is outer and outer.resolved() is outer
loop = Entity("namespace alias", "q", root)
loop.target = loop
assert loop.resolved() is loop  # an alias of itself ends the chain
assert root.qualified(["o", "Inner", "x"]) == [field] and root.qualified(["p", "Inner", "x"]) == [field]
assert root.qualified([]) == [] and root.qualified(["x"]) == [] and root.qualified(["missing", "x"]) == []
assert root.qualified(["outer", "Inner"], from_global=True) == [inner] and inner.scope.root() is root
root.declare(field, "y")
root.declare(field, "y")
root.declare(Entity("namespace", None, root))
assert root.names["y"] == [field] and root.entities() == [outer, alias, again, field]
base = Scope("class", None, None)
derived = Scope("class", None, base)
base.declare(Entity("field", "b", base))
derived.bases.append(base)
assert derived.lookup("b") == base.names["b"] and derived.lookup("c") == []
first, second = Scope("namespace", None, None), Scope("namespace", None, None)
first.transparent.append(second)
second.transparent.append(first)
second.declare(Entity("variable", "v", second))
assert first.lookup("v") == second.names["v"] and first.lookup("w") == []  # cycles end
used = Scope("namespace", None, None)
used.declare(Entity("variable", "u", used))
block = Scope("block", None, root)
block.using.append(used)
assert block.resolve("u") == used.names["u"] and block.resolve("outer") == [outer] and block.resolve("none") == []
program = Program(root)
assert program.add(outer) is outer and list(program.entities()) == [outer]
node = S.Identifier(spelling="n")
program.declares(node, outer)
program.located(node, block)
program.located(node, root)  # the first scope recorded stays
assert program.entity_of(node) is outer and program.scope_of(node) is block and program.entity_of(root) is None
assert program.lookup("outer::Inner") == [inner] and program.lookup("::outer") == [outer]
assert program.lookup("u", at=node) == used.names["u"] and program.lookup("u") == []
assert program.scope_of(S.Identifier(spelling="elsewhere")) is None and program.macros.kind == "macros"
print("ok")

## DEF-02 · Namespaces: reopened, nested, inline, unnamed, aliased, used

In [ ]:
unit, program = define("namespace a { int x; } namespace a { int y; } namespace a::b::inline c { int z; } "
                       "inline namespace v1 { int w; } namespace { int hidden; } namespace { int also; } "
                       "namespace n = a::b; namespace m = missing; namespace q = n; "
                       "namespace u { using namespace a; using namespace ::a; int k = x; } "
                       "using a::x; using none::y; enum class E { R, G }; void f() { using enum E; using enum E; R; } "
                       "namespace [[deprecated]] old {} using enum a;")
a = program.lookup("a")[0]
assert len(program.lookup("a")) == 1 and len(a.declarations) == 3 and a.scope.lookup("y")[0].kind == "variable"
assert program.lookup("a::b::z")[0].qualified_name() == "a::b::c::z"  # through the inline namespace
assert program.lookup("w")[0].qualified_name() == "v1::w" and program.lookup("hidden")[0].kind == "variable"
unnamed = [e for e in program.entities() if e.kind == "namespace" and e.name is None]
assert len(unnamed) == 1 and len(unnamed[0].declarations) == 2 and program.lookup("also")
assert program.entity_of(unit.items[4]) is program.entity_of(unit.items[5]) is unnamed[0]
assert program.lookup("n")[0].target is program.lookup("a::b")[0] and program.lookup("m")[0].target is None
assert program.lookup("q::z")[0].name == "z" and program.lookup("n::z") == program.lookup("q::z")
assert program.lookup("u")[0].scope.using == [a.scope]
assert D.referents(program, names(unit, "x")[-2]) == program.lookup("a::x")  # `int k = x`, through the directive
assert program.root.lookup("x") == program.lookup("a::x")  # the using-declaration
using = next(e for e in program.entities() if e.kind == "using declaration")
assert using.name == "y" and using.parent is program.root
r = names(unit, "R")[-1]
assert [e.qualified_name() for e in D.referents(program, r)] == ["E::R"]  # through `using enum`
print("ok")

## DEF-03 · Classes: members, bases, out-of-line definitions, unnamed classes and friends

In [ ]:
unit, program = define("""
struct B { int b; void g(); };
struct S;
struct S : B, missing::M {
    int field;
    static int count;
    static inline int ready = 1;
    static constexpr int size = 2;
    union { int i; float f; };
    struct { int depth; } info;
    void f() const;
    void f(int);
    friend class F;
    friend void g(S);
    friend void h() {}
};
int S::count = 0;
void S::f() const { field = b + size; }
void B::g() {}
struct A::Unknown { int u; };
typedef B Alias;
struct D : Alias { template <class T> friend void t(T); };
int S::extra = 3;
struct O { struct In; };
struct O::In { int z; };
""")
s = program.lookup("S")[0]
assert len(s.declarations) == 2 and isinstance(s.definition, S.ClassSpecifier) and s.definition.body is not None
assert [(e.kind, e.name) for e in s.scope.entities()] == [
    ("field", "field"), ("variable", "count"), ("variable", "ready"), ("variable", "size"), ("field", "info"),
    ("function", "f"), ("function", "f"), ("field", "extra")]  # unnamed classes are not found by name
count = program.lookup("S::count")[0]
assert len(count.declarations) == 2 and count.definition is count.declarations[1]
assert program.lookup("S::ready")[0].definition is not None and program.lookup("S::size")[0].definition is not None
assert program.lookup("S::i")[0].kind == "field"  # an anonymous union's members are its class's
assert program.lookup("S::depth") == []
const_f = next(e for e in program.lookup("S::f") if e.signature == "() const")
assert isinstance(const_f.definition, S.FunctionDefinition) and len(const_f.declarations) == 2
assert program.lookup("B::g")[0].definition is not None
assert s.scope.bases == [program.lookup("B")[0].scope] and program.lookup("S::b")[0].parent.owner.name == "B"
assert program.lookup("F") == [] and program.lookup("g") == [] and program.lookup("h") == []  # friends declare nothing
assert program.lookup("Unknown")[0].scope.lookup("u")[0].kind == "field"  # an unknown scope: declared where it is
assert program.lookup("D")[0].scope.bases == [] and program.lookup("t") == []  # an alias is no class here
assert program.lookup("S::extra")[0].parent is s.scope  # out of line, though the class does not declare it
assert program.lookup("O::In::z")[0].kind == "field"
assert program.lookup("O::In")[0].scope.parent is program.lookup("O")[0].scope
body = [n for n in F.walk(unit) if isinstance(n, S.FunctionDefinition)][1]  # S::f
field_use = names(body, "field")[0]
assert D.referents(program, field_use) == program.lookup("S::field")  # the body is in the class's scope
assert D.referents(program, names(body, "b")[0]) == program.lookup("B::b")  # and in its bases'
print("ok")

## DEF-04 · Enumerations, aliases, concepts, templates and macros

In [ ]:
unit, program = define("""
#define LIMIT 10
#define LIMIT 11
enum Color { Red, Green = LIMIT, };
enum class Mode : int { Off,
#ifdef X
  On,
#endif
};
enum Fwd : int;
typedef int Int, *IntPointer;
using Real = double;
template <typename T> concept Small = sizeof(T) < 4;
template <class T, int N, template <class> class W, typename... Ts>
struct Box { T items[N]; };
template <class, int> struct Unnamed;
template <class T> struct Box<T *> {};
template <class T> T identity(T t) { return t; }
template <class T> using Ptr = T *;
enum { Loose };
""")
assert [e.kind for e in program.macros.entities()] == ["macro"] and len(program.macros.names["LIMIT"][0].declarations) == 2
assert program.lookup("Red")[0].qualified_name() == "Color::Red" and program.lookup("Off") == []
assert program.lookup("Mode::On")[0].kind == "enumerator" and program.lookup("Fwd")[0].definition is None
assert [e.kind for e in program.lookup("Int") + program.lookup("IntPointer") + program.lookup("Real")] == [
    "type alias"] * 3
assert program.lookup("Small")[0].kind == "concept"
box = program.lookup("Box")[0]
assert len(box.declarations) == 2 and box.definition.body.items  # the partial specialization is a declaration
template = unit.items[8]
assert program.entity_of(template) is box
parameters = box.scope.parent
assert parameters.kind == "template" and [e.name for e in parameters.entities()] == ["T", "N", "W", "Ts"]
assert D.referents(program, names(template, "T")[-1])[0].kind == "template parameter"
assert program.entity_of(unit.items[11]) is program.lookup("identity")[0]
assert program.entity_of(unit.items[12]) is program.lookup("Ptr")[0]
assert program.scope_of(unit.items[9].parameters[0]).kind == "template"
assert program.entity_of(unit.items[7]) is program.lookup("Small")[0]
assert program.lookup("Loose")[0].parent.owner.name is None  # an unnamed enumeration's
print("ok")

## DEF-05 · Functions: overloads, parameters, blocks, statement scopes, labels, lambdas and handlers

In [ ]:
unit, program = define("""
void f(int);
void f(int x) { }
void f(double, ...);
int (*pointer)(int);
extern int declared;
int declared = 1;
auto [first, second] = pair;
void body(int a, int) {
    int a2 = a;
    { int inner = a2; }
    if (int c = a; c) { int d = c; }
    for (int i = 0; i < a; ++i) {}
    for (auto [k, v] : map) {}
start:
    try { } catch (const E &e) { use(e); } catch (...) {}
    auto l = [a, n = 1]<typename T, int K, class>(T t, int) { return t + n + a; };
    auto r = requires (int q) { q; };
    goto start;
}
""")
f = program.lookup("f")
assert [(e.signature, len(e.declarations), e.definition is not None) for e in f] == [
    ("(int)", 2, True), ("(double, ...)", 1, False)]
assert program.lookup("pointer")[0].kind == "variable"  # a pointer to a function is an object
declared = program.lookup("declared")[0]
assert len(declared.declarations) == 2 and declared.definition is declared.declarations[1]
assert [e.kind for e in program.lookup("first") + program.lookup("second")] == ["variable", "variable"]
body = program.lookup("body")[0]
function_scope = program.scope_of(unit.items[-1].body)
assert function_scope.kind == "function" and function_scope.owner is body
assert [(e.kind, e.name) for e in function_scope.entities()] == [("parameter", "a"), ("label", "start")]
kinds = {}
for name in ("a2", "inner", "c", "d", "i", "k", "v", "e", "n", "T", "K", "t", "q"):
    found = [e for e in program.entities() if e.name == name]
    kinds[name] = (found[0].kind, found[0].parent.kind)
assert kinds == {"a2": ("variable", "block"), "inner": ("variable", "block"), "c": ("variable", "block"),
                 "d": ("variable", "block"), "i": ("variable", "block"), "k": ("variable", "block"),
                 "v": ("variable", "block"), "e": ("variable", "block"), "n": ("variable", "lambda"),
                 "T": ("template parameter", "lambda"), "K": ("template parameter", "lambda"),
                 "t": ("parameter", "lambda"), "q": ("parameter", "requires")}
lambda_ = next(n for n in F.walk(unit) if isinstance(n, S.LambdaExpression))
uses = names(lambda_.body, "a")
assert D.referents(program, uses[0]) == [e for e in program.entities() if e.kind == "parameter" and e.name == "a"]
assert D.referents(program, S.Identifier(spelling="x")) == []  # a name outside the program
handmade = S.TranslationUnit(items=[
    S.SimpleDeclaration(specifiers=[S.PrimitiveTypeSpecifier(keyword="int")], declarators=[S.InitDeclarator(
        bitfield=S.IntegerLiteral(spelling="3"))]),
    S.FunctionDefinition(declarator=S.IdDeclarator(name=S.Identifier(spelling="g")), body=S.CompoundStatement()),
    S.SimpleDeclaration(specifiers=[S.PrimitiveTypeSpecifier(keyword="int")], declarators=[S.InitDeclarator(
        declarator=S.IdDeclarator(name=S.QualifiedName(qualifiers=[S.DecltypeSpecifier(expression=S.ThisExpression())],
                                                       name=S.Identifier(spelling="m"))))])])
built = D.define(handmade)
assert [(e.kind, e.qualified_name(), e.signature) for e in built.entities()] == [("function", "g", None),
                                                                                 ("variable", "m", None)]
# a declarator that declares no name declares nothing, a definition without a function declarator has no signature,
# and a name qualified by a decltype is declared where it is
print("ok")

## DEF-06 · Names resolve from where they are, but members after `.` and `->` do not

In [ ]:
unit, program = define("""
int x;
namespace ns { int y; }
struct P { int x; template <class T> T get(); ~P(); };
void f(P *p, P q) {
    int z = p->x + x + q.template get<P>() + ns::y + ::x + p->P::x + decltype(q)::x;
    p->~P();
}
""")
member, plain = names(unit, "x")[2:4]  # `p->x`, then `x`
assert program.scope_of(member) is None and D.referents(program, member) == []
assert D.referents(program, plain) == program.lookup("x") and program.lookup("x")[0].parent is program.root
qualified = [n for n in F.walk(unit) if isinstance(n, S.QualifiedName)]
assert [[e.qualified_name() for e in D.referents(program, q)] for q in qualified] == [["ns::y"], ["x"], [], []]
assert program.scope_of(qualified[2]) is None  # `p->P::x`: a member
template = next(n for n in F.walk(unit) if isinstance(n, S.TemplateId))
assert program.scope_of(template) is None and program.scope_of(template.arguments[0]) is not None
destructor = [n for n in F.walk(unit) if isinstance(n, S.DestructorName)][-1]
assert D.referents(program, destructor.type) == program.lookup("P")
print("ok")

## DEF-07 · Every entity of the conformance source, and every syntax node located

In [ ]:
source = open("../../conformance/sources/cpp20.cpp", encoding="utf-8").read()
unit = Ccpp20.parse(source)
program = D.define(unit)
assert len(list(program.entities())) > 100
assert program.lookup("geo::v1::Point::x")[0].kind == "field"
assert program.lookup("geo::Point") == program.lookup("geo::v1::Point")  # v1 is inline
unlocated = [n.KIND for n in F.walk(unit) if program.scope_of(n) is None]
assert set(unlocated) == {"Identifier", "TemplateId", "DestructorName"}, set(unlocated)  # members after . and ->
assert D.name_of(S.DestructorName(type=S.DecltypeSpecifier(expression=S.ThisExpression()))) == "~decltype(this)"
assert D.name_of(S.ConversionName(type=S.TypeId(specifiers=[S.PrimitiveTypeSpecifier(keyword="int")]))) == \
    "operator int"
print("ok")